# ITEM 1 & 1A EXTRACTION FROM 10-K FILINGS

**Purpose:** Extract Business (Item 1) and Risk Factors (Item 1A) sections from 10-K filings

**What This Does:**
- Extracts Item 1 (Business Description) and Item 1A (Risk Factors)
- Creates separate JSON files from MD&A extractions
- Organizes by year subfolders (2010-2025)
- Generates analysis-ready CSV and Parquet files

**Prerequisites:**
- Repository: `/content/drive/MyDrive/EDGAR_Project/edgar-crawler`
- Raw 10-K files already downloaded in `datasets/RAW_FILINGS/10-K/`
- Metadata file: `datasets/FILINGS_METADATA.csv`
- Year filter: 2010 onwards (~77,000 filings)

**Output Structure:**
```
datasets/EXTRACTED_FILINGS/
├── 10-K/              # Existing MD&A (Item 7) - UNTOUCHED
└── item_1_1a/         # NEW - Items 1 & 1A
    ├── 2010/
    ├── 2011/
    └── ...
```

---


## SECTION 1: SETUP (Run Every Time)

Run these cells at the start of every Colab session


In [ ]:
## 🟢 Cell 1: Mount Google Drive
import os
from google.colab import drive

if os.path.exists('/content/drive/MyDrive'):
    print("✅ Drive already mounted")
else:
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully")


In [ ]:
## 🟢 Cell 2: Navigate to Repository
import os

REPO_DIR = '/content/drive/MyDrive/EDGAR_Project/edgar-crawler'

if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"✅ Working directory: {os.getcwd()}")
else:
    print(f"❌ Repository not found at: {REPO_DIR}")


In [ ]:
## 🟢 Cell 3: Install Dependencies
print("📦 Installing dependencies...")

!pip install -q 'dill<0.3.9' 'multiprocess<0.70.17'
!pip install -q pox ppft
!pip install -q --no-deps pathos
!pip install -q beautifulsoup4 lxml requests pandas tqdm click cssutils numpy pyarrow

print("✅ All dependencies installed")

In [ ]:
## 🟢 Cell 4: Keep-Alive Script
from IPython.display import display, Javascript

display(Javascript('''
function KeepClicking(){
    console.log("Keeping session alive...");
    document.querySelector("colab-connect-button").click();
}
setInterval(KeepClicking, 60000);
'''))

print("✅ Keep-alive activated (prevents disconnection)")

---

## SECTION 2: CONFIGURATION

Create extraction configuration for Items 1 and 1A

In [ ]:
## ⚙️ Cell 5: Create Extraction Config
import json
import os

config_dir = 'extraction_configs'
os.makedirs(config_dir, exist_ok=True)

config_path = os.path.join(config_dir, 'items_1_1a.json')

config = {
    "description": "Extract Business (Item 1) and Risk Factors (Item 1A) from 10-K filings",
    "items_to_extract": ["1", "1A"],
    "filing_types": ["10-K"],
    "output_dir": "item_1_1a",
    "remove_tables": True,
    "skip_existing": True,
    "include_signature": False
}

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print("✅ Configuration created successfully!")
print(f"\n📄 Config file: {config_path}")
print(f"\n📋 Configuration:")
print(json.dumps(config, indent=2))

---

## SECTION 3: FILTER METADATA

Load and filter metadata to only process filings from 2010 onwards

In [ ]:
## 🔍 Cell 6: Load and Filter Metadata
import pandas as pd

metadata_path = 'datasets/FILINGS_METADATA.csv'

print("📊 Loading metadata...")

if not os.path.exists(metadata_path):
    print(f"❌ Metadata file not found: {metadata_path}")
else:
    metadata = pd.read_csv(metadata_path)
    print(f"✅ Loaded {len(metadata):,} total filings")
    
    # Filter to 10-K and 2010 onwards
    metadata_10k = metadata[metadata['Type'] == '10-K'].copy()
    metadata_filtered = metadata_10k[metadata_10k['year'] >= 2010].copy()
    
    print(f"\n📊 Filtered Metadata:")
    print(f"   Filings >= 2010: {len(metadata_filtered):,}")
    
    # Save filtered metadata
    filtered_path = 'datasets/FILINGS_METADATA_2010_onwards.csv'
    metadata_filtered.to_csv(filtered_path, index=False)
    print(f"\n✅ Filtered metadata saved: {filtered_path}")

---

## SECTION 4: CREATE OUTPUT DIRECTORY STRUCTURE

Create the new `item_1_1a/` directory with year subfolders

In [ ]:
## 📁 Cell 7: Create Output Directory Structure
import os

output_base = 'datasets/EXTRACTED_FILINGS/item_1_1a'

print("📁 Creating output directory structure...\n")

# Create base directory and year subfolders (2010-2025)
os.makedirs(output_base, exist_ok=True)
for year in range(2010, 2026):
    year_dir = os.path.join(output_base, str(year))
    os.makedirs(year_dir, exist_ok=True)

print(f"✅ Directory structure created: {output_base}/")
print(f"   Year folders: 2010-2025")

---

## SECTION 5: RUN EXTRACTION

Extract Items 1 and 1A from 10-K filings (2010 onwards)

In [ ]:
## 🚀 Cell 8: Update config.json with Filtered Metadata
import json

# Load main config
with open('config.json', 'r') as f:
    config = json.load(f)

# Update to use filtered metadata
config['extract_items']['filings_metadata_file'] = 'FILINGS_METADATA_2010_onwards.csv'
config['extract_items']['filing_types'] = ['10-K']

# Save updated config
with open('config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("✅ Main config.json updated")

In [ ]:
## 🚀 Cell 9: Run Extraction
print("="*70)
print(" STARTING EXTRACTION: ITEMS 1 & 1A")
print("="*70)
print(f"\n📊 Configuration:")
print(f"   Items: 1 (Business), 1A (Risk Factors)")
print(f"   Output: datasets/EXTRACTED_FILINGS/item_1_1a/")
print(f"   Years: 2010-2025")
print(f"\n⏱️  Estimated time: 8-12 hours\n")
print("="*70)

!python flexible_extractor.py --config extraction_configs/items_1_1a.json

---

## SECTION 6: CHECK PROGRESS

Monitor extraction progress and verify output quality

In [ ]:
## 📊 Cell 10: Check Extraction Progress
import os
import json
import pandas as pd
from collections import defaultdict

extracted_dir = 'datasets/EXTRACTED_FILINGS/item_1_1a'

print("📊 Scanning for extracted files...\n")

# Count files by year
year_counts = defaultdict(int)
all_files = []

for root, dirs, files in os.walk(extracted_dir):
    json_files = [f for f in files if f.endswith('.json')]
    if json_files and root != extracted_dir:
        year = os.path.basename(root)
        year_counts[year] = len(json_files)
        all_files.extend([os.path.join(root, f) for f in json_files])

# Load expected count
metadata = pd.read_csv('datasets/FILINGS_METADATA_2010_onwards.csv')
expected = len(metadata)

print("="*70)
print(" EXTRACTION PROGRESS")
print("="*70)
print(f"\n📊 Overall Progress:")
print(f"   Extracted: {len(all_files):,} filings")
print(f"   Expected: {expected:,} filings")
print(f"   Progress: {len(all_files)/expected*100:.1f}%")
print(f"   Remaining: {expected - len(all_files):,} filings")

# Display by year
if year_counts:
    print(f"\n📅 Files by Year:")
    for year in sorted(year_counts.keys()):
        print(f"      {year}: {year_counts[year]:,} files")

---

## SECTION 7: CREATE ANALYSIS FILES

Generate metadata CSV and consolidated Parquet file for analysis

In [ ]:
## 💾 Cell 11: Create Metadata CSV
import os
import json
import pandas as pd
from tqdm import tqdm

print("📊 Creating metadata CSV for Items 1 & 1A...\n")

extracted_dir = 'datasets/EXTRACTED_FILINGS/item_1_1a'
metadata_records = []

for root, dirs, files in os.walk(extracted_dir):
    json_files = [f for f in files if f.endswith('.json')]
    
    for filename in tqdm(json_files, desc=f"Processing {os.path.basename(root)}", leave=False):
        filepath = os.path.join(root, filename)
        try:
            with open(filepath, 'r') as f:
                filing = json.load(f)
            
            metadata_records.append({
                'filename': filename,
                'cik': filing.get('cik', ''),
                'company': filing.get('company', ''),
                'filing_date': filing.get('filing_date', ''),
                'year': filing.get('period_of_report', '')[:4],
                'has_item_1': 'item_1' in filing and len(filing.get('item_1', '')) > 0,
                'item_1_length': len(filing.get('item_1', '')),
                'has_item_1a': 'item_1a' in filing and len(filing.get('item_1a', '')) > 0,
                'item_1a_length': len(filing.get('item_1a', '')),
                'json_path': filepath
            })
        except Exception as e:
            print(f"⚠️ Error: {filename}")

# Create DataFrame and save
df_meta = pd.DataFrame(metadata_records)
meta_path = 'datasets/items_1_1a_metadata.csv'
df_meta.to_csv(meta_path, index=False)

print(f"\n✅ Metadata CSV created!")
print(f"   Location: {meta_path}")
print(f"   Records: {len(df_meta):,}")
print(f"\n📊 Summary:")
print(f"   With Item 1: {df_meta['has_item_1'].sum():,}")
print(f"   With Item 1A: {df_meta['has_item_1a'].sum():,}")

In [ ]:
## 💾 Cell 12: Create Consolidated Parquet File
import os
import json
import pandas as pd
from tqdm import tqdm

print("📦 Creating consolidated Parquet file...\n")

extracted_dir = 'datasets/EXTRACTED_FILINGS/item_1_1a'
full_data = []

for root, dirs, files in os.walk(extracted_dir):
    json_files = [f for f in files if f.endswith('.json')]
    
    for filename in tqdm(json_files, desc=f"Processing {os.path.basename(root)}", leave=False):
        filepath = os.path.join(root, filename)
        try:
            with open(filepath, 'r') as f:
                filing = json.load(f)
            
            if ('item_1' in filing and len(filing.get('item_1', '')) > 0) or \
               ('item_1a' in filing and len(filing.get('item_1a', '')) > 0):
                full_data.append({
                    'cik': filing.get('cik', ''),
                    'company': filing.get('company', ''),
                    'filing_date': filing.get('filing_date', ''),
                    'year': filing.get('period_of_report', '')[:4],
                    'item_1_text': filing.get('item_1', ''),
                    'item_1a_text': filing.get('item_1a', '')
                })
        except Exception as e:
            pass

# Create DataFrame and save
df_full = pd.DataFrame(full_data)
parquet_path = 'datasets/items_1_1a_full.parquet'
df_full.to_parquet(parquet_path, compression='gzip', index=False)

print(f"\n✅ Parquet file created!")
print(f"   Location: {parquet_path}")
print(f"   Records: {len(df_full):,}")
print(f"   File size: {os.path.getsize(parquet_path) / (1024**2):.1f} MB")

---

## SECTION 8: RESUME HELPER

Quick cells to resume extraction after disconnection

In [ ]:
## 🔄 Cell 13: Quick Resume (Run After Disconnection)
from google.colab import drive
from IPython.display import display, Javascript
import os

print("🔄 Quick Resume After Disconnection\n")

# 1. Remount Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
print("✅ Drive mounted")

# 2. Navigate to repo
os.chdir('/content/drive/MyDrive/EDGAR_Project/edgar-crawler')
print(f"✅ Working directory: {os.getcwd()}")

# 3. Install dependencies
print("\n📦 Installing dependencies...")
!pip install -q 'dill<0.3.9' 'multiprocess<0.70.17'
!pip install -q pox ppft
!pip install -q --no-deps pathos
!pip install -q beautifulsoup4 lxml requests pandas tqdm click cssutils numpy pyarrow
print("✅ Dependencies installed")

# 4. Keep-alive
display(Javascript('''
function KeepClicking(){
    console.log("Keeping session alive...");
    document.querySelector("colab-connect-button").click();
}
setInterval(KeepClicking, 60000);
'''))
print("✅ Keep-alive activated")

print("\n🎉 Ready to resume!")

In [ ]:
## 🔄 Cell 14: Resume Extraction
print("🚀 Resuming extraction...\n")
!python flexible_extractor.py --config extraction_configs/items_1_1a.json

---

## 🎉 EXTRACTION COMPLETE!

### Summary of Outputs:

1. **JSON Files** (organized by year):
   - Location: `datasets/EXTRACTED_FILINGS/item_1_1a/`
   - Structure: Year subfolders (2010-2025)
   - Each file contains: `item_1` (Business) and `item_1a` (Risk Factors)

2. **Metadata CSV**:
   - Location: `datasets/items_1_1a_metadata.csv`
   - Contains: File info, lengths, flags for Items 1 & 1A

3. **Parquet File** (compressed):
   - Location: `datasets/items_1_1a_full.parquet`
   - Contains: Full text for Items 1 & 1A
   - Use for: Text analysis, NLP, machine learning

### Next Steps:

- **Perform text analysis** (sentiment, readability, etc.)
- **Compare with MD&A data** (Item 7)
- **Research applications**: Risk analysis, business model comparison

---

In [ ]:
## 🟢 Cell 3: Install Dependencies
print("📦 Installing dependencies...")

!pip install -q 'dill<0.3.9' 'multiprocess<0.70.17'
!pip install -q pox ppft
!pip install -q --no-deps pathos
!pip install -q beautifulsoup4 lxml requests pandas tqdm click cssutils numpy pyarrow

print("✅ All dependencies installed")

In [ ]:
## 🟢 Cell 4: Keep-Alive Script
from IPython.display import display, Javascript

display(Javascript('''
function KeepClicking(){
    console.log("Keeping session alive...");
    document.querySelector("colab-connect-button").click();
}
setInterval(KeepClicking, 60000);
'''))

print("✅ Keep-alive activated (prevents disconnection)")

---

## SECTION 2: CONFIGURATION

Create extraction configuration for Items 1 and 1A

In [ ]:
## ⚙️ Cell 5: Create Extraction Config
import json
import os

config_dir = 'extraction_configs'
os.makedirs(config_dir, exist_ok=True)

config_path = os.path.join(config_dir, 'items_1_1a.json')

config = {
    "description": "Extract Business (Item 1) and Risk Factors (Item 1A) from 10-K filings",
    "items_to_extract": ["1", "1A"],
    "filing_types": ["10-K"],
    "output_dir": "item_1_1a",
    "remove_tables": True,
    "skip_existing": True,
    "include_signature": False
}

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print("✅ Configuration created successfully!")
print(f"\n📄 Config file: {config_path}")
print(f"\n📋 Configuration:")
print(json.dumps(config, indent=2))

---

## SECTION 3: FILTER METADATA

Load and filter metadata to only process filings from 2010 onwards

In [ ]:
## 🔍 Cell 6: Load and Filter Metadata
import pandas as pd

metadata_path = 'datasets/FILINGS_METADATA.csv'

print("📊 Loading metadata...")

if not os.path.exists(metadata_path):
    print(f"❌ Metadata file not found: {metadata_path}")
else:
    metadata = pd.read_csv(metadata_path)
    print(f"✅ Loaded {len(metadata):,} total filings")
    
    # Filter to 10-K and 2010 onwards
    metadata_10k = metadata[metadata['Type'] == '10-K'].copy()
    metadata_filtered = metadata_10k[metadata_10k['year'] >= 2010].copy()
    
    print(f"\n📊 Filtered Metadata:")
    print(f"   Filings >= 2010: {len(metadata_filtered):,}")
    
    # Save filtered metadata
    filtered_path = 'datasets/FILINGS_METADATA_2010_onwards.csv'
    metadata_filtered.to_csv(filtered_path, index=False)
    print(f"\n✅ Filtered metadata saved: {filtered_path}")

---

## SECTION 4: CREATE OUTPUT DIRECTORY STRUCTURE

Create the new `item_1_1a/` directory with year subfolders

In [ ]:
## 📁 Cell 7: Create Output Directory Structure
import os

output_base = 'datasets/EXTRACTED_FILINGS/item_1_1a'

print("📁 Creating output directory structure...\n")

# Create base directory and year subfolders (2010-2025)
os.makedirs(output_base, exist_ok=True)
for year in range(2010, 2026):
    year_dir = os.path.join(output_base, str(year))
    os.makedirs(year_dir, exist_ok=True)

print(f"✅ Directory structure created: {output_base}/")
print(f"   Year folders: 2010-2025")

---

## SECTION 5: RUN EXTRACTION

Extract Items 1 and 1A from 10-K filings (2010 onwards)

In [ ]:
## 🚀 Cell 8: Update config.json with Filtered Metadata
import json

# Load main config
with open('config.json', 'r') as f:
    config = json.load(f)

# Update to use filtered metadata
config['extract_items']['filings_metadata_file'] = 'FILINGS_METADATA_2010_onwards.csv'
config['extract_items']['filing_types'] = ['10-K']

# Save updated config
with open('config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("✅ Main config.json updated")

In [ ]:
## 🚀 Cell 9: Run Extraction
print("="*70)
print(" STARTING EXTRACTION: ITEMS 1 & 1A")
print("="*70)
print(f"\n📊 Configuration:")
print(f"   Items: 1 (Business), 1A (Risk Factors)")
print(f"   Output: datasets/EXTRACTED_FILINGS/item_1_1a/")
print(f"   Years: 2010-2025")
print(f"\n⏱️  Estimated time: 8-12 hours\n")
print("="*70)

!python flexible_extractor.py --config extraction_configs/items_1_1a.json

---

## SECTION 6: CHECK PROGRESS

Monitor extraction progress and verify output quality

In [ ]:
## 📊 Cell 10: Check Extraction Progress
import os
import json
import pandas as pd
from collections import defaultdict

extracted_dir = 'datasets/EXTRACTED_FILINGS/item_1_1a'

print("📊 Scanning for extracted files...\n")

# Count files by year
year_counts = defaultdict(int)
all_files = []

for root, dirs, files in os.walk(extracted_dir):
    json_files = [f for f in files if f.endswith('.json')]
    if json_files and root != extracted_dir:
        year = os.path.basename(root)
        year_counts[year] = len(json_files)
        all_files.extend([os.path.join(root, f) for f in json_files])

# Load expected count
metadata = pd.read_csv('datasets/FILINGS_METADATA_2010_onwards.csv')
expected = len(metadata)

print("="*70)
print(" EXTRACTION PROGRESS")
print("="*70)
print(f"\n📊 Overall Progress:")
print(f"   Extracted: {len(all_files):,} filings")
print(f"   Expected: {expected:,} filings")
print(f"   Progress: {len(all_files)/expected*100:.1f}%")
print(f"   Remaining: {expected - len(all_files):,} filings")

# Display by year
if year_counts:
    print(f"\n📅 Files by Year:")
    for year in sorted(year_counts.keys()):
        print(f"      {year}: {year_counts[year]:,} files")

---

## SECTION 7: CREATE ANALYSIS FILES

Generate metadata CSV and consolidated Parquet file for analysis

In [ ]:
## 💾 Cell 11: Create Metadata CSV
import os
import json
import pandas as pd
from tqdm import tqdm

print("📊 Creating metadata CSV for Items 1 & 1A...\n")

extracted_dir = 'datasets/EXTRACTED_FILINGS/item_1_1a'
metadata_records = []

for root, dirs, files in os.walk(extracted_dir):
    json_files = [f for f in files if f.endswith('.json')]
    
    for filename in tqdm(json_files, desc=f"Processing {os.path.basename(root)}", leave=False):
        filepath = os.path.join(root, filename)
        try:
            with open(filepath, 'r') as f:
                filing = json.load(f)
            
            metadata_records.append({
                'filename': filename,
                'cik': filing.get('cik', ''),
                'company': filing.get('company', ''),
                'filing_date': filing.get('filing_date', ''),
                'year': filing.get('period_of_report', '')[:4],
                'has_item_1': 'item_1' in filing and len(filing.get('item_1', '')) > 0,
                'item_1_length': len(filing.get('item_1', '')),
                'has_item_1a': 'item_1a' in filing and len(filing.get('item_1a', '')) > 0,
                'item_1a_length': len(filing.get('item_1a', '')),
                'json_path': filepath
            })
        except Exception as e:
            print(f"⚠️ Error: {filename}")

# Create DataFrame and save
df_meta = pd.DataFrame(metadata_records)
meta_path = 'datasets/items_1_1a_metadata.csv'
df_meta.to_csv(meta_path, index=False)

print(f"\n✅ Metadata CSV created!")
print(f"   Location: {meta_path}")
print(f"   Records: {len(df_meta):,}")
print(f"\n📊 Summary:")
print(f"   With Item 1: {df_meta['has_item_1'].sum():,}")
print(f"   With Item 1A: {df_meta['has_item_1a'].sum():,}")

In [ ]:
## 💾 Cell 12: Create Consolidated Parquet File
import os
import json
import pandas as pd
from tqdm import tqdm

print("📦 Creating consolidated Parquet file...\n")

extracted_dir = 'datasets/EXTRACTED_FILINGS/item_1_1a'
full_data = []

for root, dirs, files in os.walk(extracted_dir):
    json_files = [f for f in files if f.endswith('.json')]
    
    for filename in tqdm(json_files, desc=f"Processing {os.path.basename(root)}", leave=False):
        filepath = os.path.join(root, filename)
        try:
            with open(filepath, 'r') as f:
                filing = json.load(f)
            
            if ('item_1' in filing and len(filing.get('item_1', '')) > 0) or \
               ('item_1a' in filing and len(filing.get('item_1a', '')) > 0):
                full_data.append({
                    'cik': filing.get('cik', ''),
                    'company': filing.get('company', ''),
                    'filing_date': filing.get('filing_date', ''),
                    'year': filing.get('period_of_report', '')[:4],
                    'item_1_text': filing.get('item_1', ''),
                    'item_1a_text': filing.get('item_1a', '')
                })
        except Exception as e:
            pass

# Create DataFrame and save
df_full = pd.DataFrame(full_data)
parquet_path = 'datasets/items_1_1a_full.parquet'
df_full.to_parquet(parquet_path, compression='gzip', index=False)

print(f"\n✅ Parquet file created!")
print(f"   Location: {parquet_path}")
print(f"   Records: {len(df_full):,}")
print(f"   File size: {os.path.getsize(parquet_path) / (1024**2):.1f} MB")

---

## SECTION 8: RESUME HELPER

Quick cells to resume extraction after disconnection

In [ ]:
## 🔄 Cell 13: Quick Resume (Run After Disconnection)
from google.colab import drive
from IPython.display import display, Javascript
import os

print("🔄 Quick Resume After Disconnection\n")

# 1. Remount Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
print("✅ Drive mounted")

# 2. Navigate to repo
os.chdir('/content/drive/MyDrive/EDGAR_Project/edgar-crawler')
print(f"✅ Working directory: {os.getcwd()}")

# 3. Install dependencies
print("\n📦 Installing dependencies...")
!pip install -q 'dill<0.3.9' 'multiprocess<0.70.17'
!pip install -q pox ppft
!pip install -q --no-deps pathos
!pip install -q beautifulsoup4 lxml requests pandas tqdm click cssutils numpy pyarrow
print("✅ Dependencies installed")

# 4. Keep-alive
display(Javascript('''
function KeepClicking(){
    console.log("Keeping session alive...");
    document.querySelector("colab-connect-button").click();
}
setInterval(KeepClicking, 60000);
'''))
print("✅ Keep-alive activated")

print("\n🎉 Ready to resume!")

In [ ]:
## 🔄 Cell 14: Resume Extraction
print("🚀 Resuming extraction...\n")
!python flexible_extractor.py --config extraction_configs/items_1_1a.json

---

## 🎉 EXTRACTION COMPLETE!

### Summary of Outputs:

1. **JSON Files** (organized by year):
   - Location: `datasets/EXTRACTED_FILINGS/item_1_1a/`
   - Structure: Year subfolders (2010-2025)
   - Each file contains: `item_1` (Business) and `item_1a` (Risk Factors)

2. **Metadata CSV**:
   - Location: `datasets/items_1_1a_metadata.csv`
   - Contains: File info, lengths, flags for Items 1 & 1A

3. **Parquet File** (compressed):
   - Location: `datasets/items_1_1a_full.parquet`
   - Contains: Full text for Items 1 & 1A
   - Use for: Text analysis, NLP, machine learning

### Next Steps:

- **Perform text analysis** (sentiment, readability, etc.)
- **Compare with MD&A data** (Item 7)
- **Research applications**: Risk analysis, business model comparison

---